<a href="https://colab.research.google.com/github/ellianorsX/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install transformers torch bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.7 MB/s  0:00:00


In [16]:
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch

In [21]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

#print('Vocab size:', tokenizer.vocab_size)
#print('Special tokens:'), tokenizer.all_special_tokens

# quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

bnb_config

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    #torch_dtype=torch.bfloat16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [23]:
# Before Fine-tuning
#prompt = "Explain what a tokenizer is."
prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)  # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
    )

print(
    tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
    )
)

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A tokenizer is a tool in natural language processing that splits text into tokens, or parts of words. It's used for text processing and machine learning. Text tokenization is a vital step in processing large corpora into useful data.

2. NLP (Natural Language Processing)
NLP, or Natural Language Processing, is a branch of computer science that focuses on how computers understand and generate human language. The


In [24]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


total_params = count_parameters(model)

print(
    f"Total parameters (including frozen 4-bit): "
    f"{total_params:,}"
)

Total parameters (including frozen 4-bit): 615,606,272


## dataset library | load dataset
### instruction tuning

In [28]:
from datasets import load_dataset

dataset = load_dataset(
    "yahma/alpaca-cleaned",
    split="train",
)

dataset

#dataset[1]

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 51760
})

In [30]:
def generate_prompt(example):
    instruction = example["instruction"]
    input_text = example["input"]
    output_text = example["output"]

    if input_text:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Input:\n"
            f"{input_text}\n\n"
            "### Response:\n"
            f"{output_text}"
        )
    else:
        return (
            "### Instruction:\n"
            f"{instruction}\n\n"
            "### Response:\n"
            f"{output_text}"
        )


#generate_prompt(dataset[0])

def formatting_func(example):
  return {'text': generate_prompt(example)}

dataset = dataset.map(formatting_func)



Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [32]:
dataset[0] ['text']

'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [33]:
dataset = dataset.select(range(7000))

In [34]:
dataset = dataset.shuffle(seed=42)

In [35]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 7000
})